# Patient Health Outcome Predictor: model training notebook

This notebook trains the model that powers the web app. Run it **from top to bottom**, one cell at a time. When it finishes it saves one file, `../backend/model/outcome_model.joblib`, and the web app picks that file up automatically.

**What we predict:** the chance of a bad outcome for a patient, from their demographics, medical history and treatment details. In this example the outcome is *"Readmission to hospital within 30 days"*.

**Default dataset:** [Diabetes 130-US Hospitals for Years 1999-2008](https://archive.ics.uci.edu/dataset/296/diabetes-130-us-hospitals-for-years-1999-2008) (UCI Machine Learning Repository, 101,766 hospital visits, licence CC BY 4.0). It has demographics, medical history, treatment details and a readmission label, which is exactly what this project needs.

| Step | What happens |
|---|---|
| 1 | Import libraries |
| 2 | Settings (the only cell you edit to change dataset or features) |
| 3 | Load the data |
| 4 | Explore the raw data |
| 5 | Clean the data and create the target |
| 6 | Look at the cleaned data |
| 7 | Choose features and split into train / test |
| 8 | Build the preprocessing pipeline |
| 9 | Compare models with cross-validation |
| 10 | Evaluate the best model on the test set |
| 11 | See which features matter most |
| 12 | Save the model for the web app |
| 13 | Check the saved model |

**Two notes before you start**

- *Race is left out of the inputs on purpose.* Using it in a health-risk model can build unfair bias into predictions. You can add it back in Step 2 if you want to study it.
- *This is a learning and demo project.* It is not a medical device and must not be used for real clinical decisions.

## Step 1: Import libraries

If something is missing, install it from a terminal with `pip install -r requirements.txt` (the file is in this folder).

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 100

print("scikit-learn version:", sklearn.__version__)

## Step 2: Settings

Everything you might want to change is in this one cell.

- `NUMERIC_FEATURES` / `CATEGORICAL_FEATURES` decide which columns become inputs in the web form.
- `FEATURE_LABELS` are the friendly names shown in the web form.
- `RISK_PERCENTILES = (60, 90)` means: the **top 10%** of predicted risk is called *High*, the next 30% *Moderate*, the rest *Low*. These bands are relative to the patients in your data, because an outcome like readmission is rare and a fixed cut-off such as 50% would almost never trigger.

**Using a different dataset?** Change `DATA_FILE`, `RAW_TARGET_COL`, `OUTCOME_LABEL` and the feature lists here, then adjust the cleaning code in Step 5. Steps 6 to 13 stay the same.

In [ ]:
# ======================= SETTINGS =======================
DATASET_NAME = "UCI Diabetes 130-US Hospitals (1999-2008)"
DATA_DIR = Path("data")
DATA_FILE = DATA_DIR / "diabetic_data.csv"
DATA_URL = ("https://archive.ics.uci.edu/static/public/296/"
            "diabetes+130-us+hospitals+for+years+1999-2008.zip")

OUTCOME_LABEL = "Readmission within 30 days"   # shown in the web app
RAW_TARGET_COL = "readmitted"                    # label column in the raw file

# Columns the web form will ask for (they must exist after the cleaning in Step 5)
NUMERIC_FEATURES = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
]
CATEGORICAL_FEATURES = [
    "age", "gender", "admission_type", "primary_diagnosis",
    "A1Cresult", "insulin", "med_change", "diabetesMed",
]

FEATURE_LABELS = {
    "age": "Age group (years)",
    "gender": "Gender",
    "admission_type": "Admission type",
    "primary_diagnosis": "Primary diagnosis group",
    "A1Cresult": "HbA1c test result",
    "insulin": "Insulin dose",
    "med_change": "Diabetes medication changed",
    "diabetesMed": "On diabetes medication",
    "time_in_hospital": "Days in hospital",
    "num_lab_procedures": "Number of lab tests",
    "num_procedures": "Number of procedures (non-lab)",
    "num_medications": "Number of medications",
    "number_outpatient": "Outpatient visits in prior year",
    "number_emergency": "Emergency visits in prior year",
    "number_inpatient": "Inpatient stays in prior year",
    "number_diagnoses": "Number of diagnoses",
}

# Optional: order of the dropdown options in the web form (otherwise: most common first)
CATEGORY_ORDER = {
    "age": ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)", "[50-60)",
            "[60-70)", "[70-80)", "[80-90)", "[90-100)"],
    "insulin": ["No", "Steady", "Up", "Down"],
    "A1Cresult": ["Not measured", "Normal", "Above 7", "Above 8"],
}

RISK_PERCENTILES = (60, 90)     # Low < 60th percentile <= Moderate < 90th <= High
TEST_SIZE = 0.2
RANDOM_STATE = 42
MODEL_OUT = Path("../backend/model/outcome_model.joblib")

## Step 3: Load the data

The cell below looks for `data/diabetic_data.csv`. If it is not there, it tries to download it from UCI. If the download fails (for example, no internet or a firewall), do it by hand:

1. Open the [dataset page](https://archive.ics.uci.edu/dataset/296/diabetes-130-us-hospitals-for-years-1999-2008) and click **Download**.
2. Unzip it and put `diabetic_data.csv` inside the `notebook/data/` folder.
3. Run the cell again.

In this file a question mark (`?`) means "missing", so we tell pandas to treat it that way.

In [ ]:
def download_dataset():
    import io
    import urllib.request
    import zipfile

    DATA_DIR.mkdir(exist_ok=True)
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        zipfile.ZipFile(io.BytesIO(response.read())).extractall(DATA_DIR)


if not DATA_FILE.exists():
    print("Dataset not found locally, trying to download it from UCI ...")
    try:
        download_dataset()
    except Exception as error:
        raise FileNotFoundError(
            f"Could not download the dataset ({error}).\n"
            f"Download it manually from the UCI page and place the CSV at: {DATA_FILE.resolve()}"
        )

raw = pd.read_csv(DATA_FILE, na_values="?", low_memory=False)
print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]} columns")
raw.head()

## Step 4: Explore the raw data

Before changing anything, look at what we have: the column types, how much is missing, and what the label looks like.

In [ ]:
print("Column types:")
print(raw.dtypes.value_counts(), "\n")

missing = raw.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("Columns with missing values (% of rows):")
print(missing[missing > 0], "\n")

print(f"Label column '{RAW_TARGET_COL}':")
print(raw[RAW_TARGET_COL].value_counts())

**What to notice:** `weight` is almost always missing, so we will not use it. The label has three values: `<30` (readmitted within 30 days), `>30` (readmitted later) and `NO` (never readmitted). We care about the first one.

## Step 5: Clean the data and create the target

Each small step below has a reason:

1. **Remove patients who died or went to hospice.** They cannot be readmitted, so they would unfairly look like "good outcomes".
2. **Remove the few rows with an invalid gender.**
3. **Keep one visit per patient.** Some patients appear many times. If the same patient is in both train and test sets, the model can "cheat" by recognising them.
4. **Make readable columns** (diagnosis groups, admission type, test result labels).
5. **Create the target** `target`: `1` if the patient was readmitted within 30 days, otherwise `0`.

In [ ]:
df = raw.copy()

# 5.1 Remove patients who died or were sent to hospice (discharge codes 11, 13, 14, 19, 20, 21)
DEAD_OR_HOSPICE = [11, 13, 14, 19, 20, 21]
before = len(df)
df = df[~df["discharge_disposition_id"].isin(DEAD_OR_HOSPICE)]
print(f"5.1 removed {before - len(df):,} visits (patient died or hospice)")

# 5.2 Remove rows with an invalid gender
before = len(df)
df = df[df["gender"].isin(["Male", "Female"])]
print(f"5.2 removed {before - len(df):,} visits (invalid gender)")

# 5.3 Keep only the first visit of each patient
before = len(df)
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
print(f"5.3 removed {before - len(df):,} repeat visits (one row per patient now)")


# 5.4 Make readable columns
def diagnosis_group(code):
    # Groups the ICD-9 diagnosis code into broad categories
    if pd.isna(code):
        return "Other"
    code = str(code)
    if code.startswith("250"):
        return "Diabetes"
    if code[0] in "VE":
        return "Other"
    try:
        n = float(code)
    except ValueError:
        return "Other"
    if 390 <= n < 460 or int(n) == 785:
        return "Circulatory"
    if 460 <= n < 520 or int(n) == 786:
        return "Respiratory"
    if 520 <= n < 580 or int(n) == 787:
        return "Digestive"
    if 580 <= n < 630 or int(n) == 788:
        return "Genitourinary"
    if 710 <= n < 740:
        return "Musculoskeletal"
    if 800 <= n < 1000:
        return "Injury"
    if 140 <= n < 240:
        return "Neoplasms"
    return "Other"


ADMISSION_TYPE = {1: "Emergency", 2: "Urgent", 3: "Elective", 4: "Newborn",
                  5: "Not available", 6: "Not available", 7: "Trauma centre", 8: "Not available"}

df["primary_diagnosis"] = df["diag_1"].apply(diagnosis_group)
df["admission_type"] = df["admission_type_id"].map(ADMISSION_TYPE).fillna("Not available")
df["A1Cresult"] = (df["A1Cresult"].fillna("Not measured")
                   .replace({"None": "Not measured", "Norm": "Normal",
                             ">7": "Above 7", ">8": "Above 8"}))
df["med_change"] = df["change"].map({"Ch": "Changed", "No": "No change"})

# 5.5 Create the target: 1 = readmitted within 30 days, 0 = not
df["target"] = (df[RAW_TARGET_COL] == "<30").astype(int)

print(f"\nClean data: {len(df):,} patients")
print(f"Readmitted within 30 days: {df['target'].mean():.1%}")

## Step 6: Look at the cleaned data

A quick look at how the readmission rate changes with a few inputs. This is a good sanity check: it should match common sense (for example, patients with more previous hospital stays tend to be readmitted more often).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

(df.groupby("age")["target"].mean() * 100).plot(kind="bar", ax=axes[0], color="#0f766e")
axes[0].set_title("Readmission rate by age group")

(df.assign(prior=df["number_inpatient"].clip(upper=5))
   .groupby("prior")["target"].mean() * 100).plot(kind="bar", ax=axes[1], color="#0f766e")
axes[1].set_title("Readmission rate by prior inpatient stays (5 = 5 or more)")
axes[1].set_xlabel("prior inpatient stays")

(df.groupby("primary_diagnosis")["target"].mean() * 100).sort_values().plot(
    kind="barh", ax=axes[2], color="#0f766e")
axes[2].set_title("Readmission rate by primary diagnosis")

for ax in axes:
    ax.set_ylabel("%" if ax is not axes[2] else "")
plt.tight_layout()
plt.show()

## Step 7: Choose features and split into train / test

We keep only the columns listed in the settings. Then we split the patients:

- **80% train:** the model learns from these.
- **20% test:** kept hidden until the very end, to give an honest score.

`stratify=y` keeps the same share of readmitted patients in both parts.

In [ ]:
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

data = df[FEATURES + ["target"]].copy()
for col in CATEGORICAL_FEATURES:
    data[col] = data[col].fillna("Unknown").astype(str)

X = data[FEATURES]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Features used: {len(FEATURES)}")
print(f"Train: {len(X_train):,} patients ({y_train.mean():.1%} readmitted)")
print(f"Test:  {len(X_test):,} patients ({y_test.mean():.1%} readmitted)")
print("\nMissing values left in the inputs:", int(X.isna().sum().sum()))

## Step 8: Build the preprocessing pipeline

Models need numbers, so we prepare each type of column:

- **Numeric columns:** fill any gaps with the median, then scale them to a similar range.
- **Text columns:** fill any gaps with the most common value, then turn each category into 0/1 columns (*one-hot encoding*). Categories the model has never seen are ignored instead of crashing.

Putting this inside a scikit-learn `Pipeline` means the web app can feed raw form values straight in, and the exact same preparation is applied.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])
preprocess

## Step 9: Compare models with cross-validation

We try two models:

- **Logistic Regression:** simple, fast and easy to explain.
- **Gradient Boosting:** more flexible, can capture interactions between inputs.

Each is scored with 5-fold cross-validation on the **training data only**. The score is **ROC-AUC**: 0.5 means guessing, 1.0 means perfect. For hospital readmission, scores around 0.60 to 0.70 are normal, because this outcome is hard to predict.

In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.08, max_depth=4, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
for name, estimator in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", estimator)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    results.append({"model": name, "mean_auc": scores.mean(), "std": scores.std()})

cv_table = pd.DataFrame(results).sort_values("mean_auc", ascending=False).reset_index(drop=True)
best_name = cv_table.loc[0, "model"]
print(cv_table.round(4).to_string(index=False))
print(f"\nBest model: {best_name}")

## Step 10: Evaluate the best model on the test set

Now we train the best model on all the training data and score it **once** on the test patients it has never seen. We look at:

- **ROC-AUC:** how well it ranks risky patients above safe ones.
- **Average precision:** useful when the outcome is rare. Compare it with the baseline (the plain readmission rate).
- **Brier score:** how close the predicted probabilities are to reality (lower is better).
- **Calibration plot:** if the model says 20%, do about 20% of those patients really have the outcome? Points close to the diagonal line are good.
- **Risk bands:** the Low / Moderate / High groups from Step 2, with the *actual* outcome rate in each. A good model shows the rate going up from Low to High.

In [ ]:
final_pipe = Pipeline([("prep", preprocess), ("model", candidates[best_name])])
final_pipe.fit(X_train, y_train)

test_proba = final_pipe.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_proba)
ap = average_precision_score(y_test, test_proba)
brier = brier_score_loss(y_test, test_proba)
baseline = float(y_test.mean())

print(f"ROC-AUC           : {auc:.3f}")
print(f"Average precision : {ap:.3f}   (baseline if guessing: {baseline:.3f})")
print(f"Brier score       : {brier:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[0].plot(fpr, tpr, color="#0f766e", label=f"{best_name} (AUC {auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Guessing")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve (test set)")
axes[0].legend()

frac_true, frac_pred = calibration_curve(y_test, test_proba, n_bins=10, strategy="quantile")
axes[1].plot(frac_pred, frac_true, "o-", color="#0f766e", label="Model")
top = max(frac_pred.max(), frac_true.max()) * 1.1
axes[1].plot([0, top], [0, top], "--", color="gray", label="Perfect")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Actual rate")
axes[1].set_title("Calibration (test set)")
axes[1].legend()
plt.tight_layout()
plt.show()

# Risk bands, based on percentiles of the predicted risk
p_moderate, p_high = np.percentile(test_proba, RISK_PERCENTILES)
band = np.where(test_proba >= p_high, "High",
                np.where(test_proba >= p_moderate, "Moderate", "Low"))
band_table = (pd.DataFrame({"band": band, "outcome": y_test.values})
              .groupby("band")["outcome"].agg(patients="size", actual_rate="mean")
              .reindex(["Low", "Moderate", "High"]))
print("\nActual outcome rate in each risk band (test set):")
print(band_table.assign(actual_rate=(band_table["actual_rate"] * 100).round(1)).to_string())

## Step 11: See which features matter most

**Permutation importance** shuffles one input at a time and measures how much the score drops. A big drop means the model relies on that input. (We use up to 5,000 test patients to keep it fast.)

In [ ]:
sample = X_test.sample(min(5000, len(X_test)), random_state=RANDOM_STATE)
perm = permutation_importance(
    final_pipe, sample, y_test.loc[sample.index],
    scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

labels = [FEATURE_LABELS.get(name, name) for name in importance.index]
plt.figure(figsize=(8, 5))
plt.barh(labels, importance.values, color="#0f766e")
plt.xlabel("Drop in ROC-AUC when the feature is shuffled")
plt.title("Feature importance (test set)")
plt.tight_layout()
plt.show()

## Step 12: Save the model for the web app

We save one file containing:

- the trained pipeline,
- a description of every input (name, label, allowed range or dropdown options, default value), so the web form builds itself,
- the scores, risk-band cut-offs and global feature importance, shown in the app.

Because the form is built from this file, **you can change the features or the dataset and the web app adapts without any code change.**

In [ ]:
def build_schema(train_frame):
    schema = []
    for name in NUMERIC_FEATURES:
        values = train_frame[name]
        is_integer = bool(np.allclose(values.dropna() % 1, 0))
        cast = int if is_integer else float
        schema.append({
            "name": name,
            "label": FEATURE_LABELS.get(name, name),
            "type": "number",
            "integer": is_integer,
            "min": cast(values.min()),
            "max": cast(values.max()),
            "default": cast(round(values.median())) if is_integer else float(values.median()),
        })
    for name in CATEGORICAL_FEATURES:
        counts = train_frame[name].value_counts()
        ordered = [o for o in CATEGORY_ORDER.get(name, []) if o in counts.index]
        ordered += [o for o in counts.index if o not in ordered]
        schema.append({
            "name": name,
            "label": FEATURE_LABELS.get(name, name),
            "type": "select",
            "options": [str(o) for o in ordered],
            "default": str(counts.idxmax()),
        })
    return schema


bundle = {
    "pipeline": final_pipe,
    "features": build_schema(X_train),
    "outcome_label": OUTCOME_LABEL,
    "dataset_name": DATASET_NAME,
    "model_name": best_name,
    "trained_rows": int(len(X_train)),
    "base_rate": float(y_train.mean()),
    "metrics": {"roc_auc": round(float(auc), 3),
                "average_precision": round(float(ap), 3),
                "brier": round(float(brier), 4),
                "test_rows": int(len(X_test))},
    "risk_bands": {
        "percentiles": list(RISK_PERCENTILES),
        "moderate_from": float(p_moderate),
        "high_from": float(p_high),
        "actual_rates": {k: round(float(v), 4) for k, v in band_table["actual_rate"].items()},
    },
    # 101 points (0th to 100th percentile) of the test-set predictions, used to say
    # "this patient is riskier than X% of patients"
    "score_quantiles": [float(v) for v in np.percentile(test_proba, np.arange(101))],
    "global_importance": [
        {"feature": FEATURE_LABELS.get(n, n), "importance": round(float(v), 5)}
        for n, v in importance.sort_values(ascending=False).items()
    ],
    "sklearn_version": sklearn.__version__,
}

MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_OUT, compress=3)

print(f"Saved: {MODEL_OUT.resolve()}  ({MODEL_OUT.stat().st_size / 1024:.0f} KB)")
print(f"\nIMPORTANT: put this exact line in the requirements.txt of the web app,")
print(f"so the server uses the same scikit-learn version as this notebook:")
print(f"    scikit-learn=={sklearn.__version__}")

## Step 13: Check the saved model

Load the file back exactly as the web app does, and predict for a typical patient (every input at its default value).

In [ ]:
loaded = joblib.load(MODEL_OUT)

example = {f["name"]: f["default"] for f in loaded["features"]}
print("Example patient:", example)

probability = loaded["pipeline"].predict_proba(pd.DataFrame([example]))[0, 1]
print(f"\nPredicted chance of '{loaded['outcome_label']}': {probability:.1%}")
print(f"Average in the training data: {loaded['base_rate']:.1%}")

## Done. What next?

1. Make sure `scikit-learn==...` in the app's `requirements.txt` matches the version printed in Step 12.
2. Start the web app (see the project README):
   ```
   pip install -r requirements.txt
   python backend/app.py
   ```
   then open http://localhost:5000.
3. To deploy, commit `backend/model/outcome_model.joblib` together with the code.

**Ideas to improve the model:** add more input columns (for example the other medication columns), tune the model settings (`GridSearchCV` or `RandomizedSearchCV`), try other models, or retrain with the full data (train + test) once you are happy with the scores.

**Use another dataset:** see the dataset list in the project README. For a table like the *Heart Failure Clinical Records* data, change `DATA_FILE`, `RAW_TARGET_COL`, `OUTCOME_LABEL` and the feature lists in Step 2, replace the cleaning in Step 5 with whatever your file needs (check `raw.columns` for the exact names), and keep Steps 6 to 13 as they are. Do not use columns that are only known after the outcome happens (for example the follow-up time in that dataset), because they would leak the answer.